In [1]:
import pandas as pd
import numpy as np
import ast
import os
from datetime import datetime, timezone

Here we find the latest folder to determine where we need to still do the preprocessing.

In [2]:
SAVED_DIR = "saved"

def parse_folder_datetime(folder_name: str) -> datetime | None:
    try:
        iso_like = folder_name.replace("T", " ").replace("Z", "")
        date_part, time_part = iso_like.split(" ")

        h, m, s, ms = time_part.split("-")
        iso_string = f"{date_part}T{h}:{m}:{s}.{ms}Z"

        return datetime.strptime(
            iso_string, "%Y-%m-%dT%H:%M:%S.%fZ"
        ).replace(tzinfo=timezone.utc)

    except Exception:
        return None


def find_latest_folder_datetime() -> datetime:
    dates = []

    for name in os.listdir(SAVED_DIR):
        path = os.path.join(SAVED_DIR, name)

        if not os.path.isdir(path):
            continue

        if not name.startswith("2"):
            continue

        dt = parse_folder_datetime(name)
        if dt:
            dates.append(dt)

    if not dates:
        raise RuntimeError("No valid dated folders found in saved/")

    return max(dates)


def to_csv_format(dt: datetime) -> str:
    """YYYY-MM-DDTHH:MM:SS.mmmZ"""
    return (
        dt.astimezone(timezone.utc)
        .strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
    )


def to_folder_format(dt: datetime) -> str:
    """YYYY-MM-DDTHH-MM-SS-mmmZ"""
    return (
        dt.astimezone(timezone.utc)
        .strftime("%Y-%m-%dT%H-%M-%S-%f")[:-3] + "Z"
    )
latest_dt = find_latest_folder_datetime()

end_date_csv = to_csv_format(latest_dt)
end_date_folder = to_folder_format(latest_dt)

output_dir = f"saved/{end_date_folder}/cve_metadata_{end_date_csv}.csv"

print("end_date_csv   =", end_date_csv)
print("end_date_folder=", end_date_folder)
print("output_dir     =", output_dir)


end_date_csv   = 2026-03-23T15:40:42.551Z
end_date_folder= 2026-03-23T15-40-42-551Z
output_dir     = saved/2026-03-23T15-40-42-551Z/cve_metadata_2026-03-23T15:40:42.551Z.csv


The code to build the KG expects a certain format of CVEs, CWEs, and CPEs, so here we are just reformatting everything. All that needs to be changed is instead of lists, the CPE and CWE need to split using a singular semicolon. Then the columns need to be renamed.

In [3]:
curr_cves = pd.read_csv(output_dir)

In [ ]:
list_columns = ["matching_cwes", "matching_cpes", "keywords"]

# string to list
for col in list_columns:
    curr_cves[col] = curr_cves[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

# join CWEs and CPEs into semicolon-separated strings
for col in ["matching_cwes", "matching_cpes"]:
    curr_cves[col] = curr_cves[col].apply(
        lambda x: ";".join(map(str, x)) if isinstance(x, (list, set)) else ""
    )


In [ ]:
curr_cves.rename(columns={
    "matching_cwes": "MatchingCWE",
    "matching_cpes": "MatchingCPE",
    "cve_id": "ID"
}, inplace=True)

In [ ]:
output_dir = f"saved/{end_date_folder}/cve_reformatted.csv"
curr_cves.to_csv(output_dir, index=False)